# 2D圧密計算 — Colab / CUDA

計算は `src/mikasa_consolidation/` に置き、このNotebookでは初期化・呼び出し・結果表示だけを行います。

Colabの「ランタイム → ランタイムのタイプを変更」でGPU（例: T4）を選び、上から順に実行してください。
ローカルのJupyterでは、先に `uv sync --extra notebook` とカーネル登録を行い、CPUで試す場合は次の `DEVICE` を `"cpu"` に変更します。
GPUを選択しただけではCUDA計算の確認になりません。実デバイス情報とCPUとの差を後半で記録します。


In [ ]:
DEVICE = "cuda"
REF = "main"
OUTPUT_DIR = "outputs/colab"
COMPARISON = dict(nx=21, nz=21, dt=5.0, total_time=100.0)


## 1. リポジトリと依存関係の準備


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Mr-Kondo/2D_MIKASA_CONSOLIDATION_THEORY.git"
if IN_COLAB:
    REPO_DIR = Path("/content/2D_MIKASA_CONSOLIDATION_THEORY")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", "FETCH_HEAD"], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO_DIR)
if IN_COLAB:
    # Colab may already provide CuPy for CUDA 13; do not install a second variant.
    needs_cupy = DEVICE == "cuda" and importlib.util.find_spec("cupy") is None
    package = ".[cuda]" if needs_cupy else "."
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", package], check=True)
# The running kernel has not reread the editable-install .pth file.
SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()
import mikasa_consolidation
print("package:", mikasa_consolidation.__file__)
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. 実行環境

CUDAが利用できなければ、このセルで停止します。CPUへの暗黙の切り替えは行いません。


In [ ]:
import json
import platform
from mikasa_consolidation.cuda import cuda_info

runtime = {"device": DEVICE, "python": platform.python_version()}
if DEVICE == "cuda":
    runtime.update(cuda_info())
print(json.dumps(runtime, indent=2))


## 3. 主要な計算とコンター図

有限ひずみモデルでは三重対角ソルバーをCUDA倍精度で解き、残差・反復・収束判定はCPU倍精度で行います。
線形モデルでは固有モードの行列演算をCUDA倍精度で行います。初期化と描画はCPUです。
小さい格子ではGPU併用が速いとは限りません。これは計算コードを実際にGPUで動かす経路であり、高速化の保証ではありません。


In [ ]:
from mikasa_consolidation.cli import run_examples

simulations = run_examples(device=DEVICE, output_dir=OUTPUT_DIR)


In [ ]:
from IPython.display import Image, display

for name in ("mikasa_consolidation_result.png", "terzaghi_consolidation_result.png",
             "terzaghi_mikasa_comparison.png"):
    display(Image(filename=str(Path(OUTPUT_DIR) / name)))


## 4. CPUとの一致と時間の比較

両方式で同じ格子・時間・材料条件を使います。初期化、転送、GPU完了待ち、CPU出力を含め、描画・インストールは含みません。
初回を別記し、その後は実行順を交互に各3回測定します。数値差が許容範囲を超えると例外で停止します。


In [ ]:
from mikasa_consolidation.validation import compare_devices

if DEVICE == "cuda":
    report = compare_devices(COMPARISON)
    report["commit"] = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    Path(OUTPUT_DIR, "cpu_cuda_comparison.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))
else:
    print("CPU execution only; CUDA comparison was not run.")


## 5. CUDAの回帰テスト

1セル・1列・非正方格子・リセット・収束失敗を検証します。


In [ ]:
if DEVICE == "cuda":
    environment = dict(os.environ, MIKASA_TEST_CUDA="1", MPLBACKEND="Agg")
    subprocess.run([sys.executable, "-m", "unittest", "tests.test_cuda", "-v"], env=environment, check=True)
else:
    print("CUDA hardware tests were not run.")


## 確認範囲と終了

生成物は `outputs/colab/` に保存されます。必要な結果をダウンロードしてから、Colabの「ランタイムを接続解除して削除」でリソースを解放してください。
CPU/CUDA間の一致は実地盤への適合性を示すものではありません。実測データ検証はIssue #32で追跡しています。
